In [ ]:
# VectorDB에 텍스트 파일 데이터를 저장 후 유사 문장 읽기
!pip install chromadb sentence-transformers

In [ ]:
import os, re
import uuid   # 고유 id 식별 생성
from typing import List
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient

TXT_PATH = "sample.txt"
CHROMA_DIR = ".txt_demo"
COLLECTION = "docs_coll"
MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)
client = PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection(COLLECTION)


In [ ]:
# 파일 읽기
def read_textFunc(path:str) -> str:
    if not os.path.exists(path):
        raise FileNotFoundError(f"파일 없음:{path}")

    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

# 문단 분리 함수
def split_paragraphFunc(text:str, min_len=20) -> List[str]:
    paras = re.split(r"\n\s*\n+", text)   # 빈 줄 기준 분리
    paras = [re.sub(r"\s+", " ", p).strip() for p in paras]
    return [p for p in paras if len(p) >= min_len]

# 임베딩 함수
def embedFunc(texts:List[str]) -> List[List[float]]:
    return model.encode(texts, normalize_embeddings=True).tolist()

# 저장 함수
def upsert_paraFunc(source_path:str):
    text = read_textFunc(source_path)   # txt 파일 읽기
    # print(text)
    chunks = split_paragraphFunc(text)  # 문단 단위 분리
    # print(chunks)
    if not chunks:
        print("저장할 문단 없음")
        return

    ids = [str(uuid.uuid4()) for _ in chunks]  # 각 문단에 고유 id 부여
    print(ids)
    embs = embedFunc(chunks)   # 각 문단을 벡터화
    print(embs)
    # 기타 정보 (파일명, 문단 길이)
    metas = [{"source":os.path.basename(source_path), "len": len(c)} for c in chunks]
    print(metas)

    collection.add(
        ids=ids,
        documents=chunks,
        embeddings=embs,
        metadatas=metas
    )

# 검색
def searchFunc(query:str, k:int):
    q_emb = embedFunc([query])
    # print(q_emb)
    res = collection.query(query_embeddings=q_emb, n_results=k)
    # print(res)

    docs = res.get("documents", [[]])[0]   # [[]] 예외 방지용 패턴
    metas = res.get("metadatas", [[]])[0]
    ids = res.get("ids", [[]])[0]
    dists = res.get("distances", [[]])[0]

    for i, (doc, meta, _id, dist) in enumerate(zip(docs, metas, ids, dists), start=1):
        print(f"\n[{i}] id={_id}")
        print(f"source={meta.get('source')}, len={meta.get('len')}, distance={dist:.4f}")
        print(doc[:500] + ("..." if len(doc) > 500 else ""))


if __name__ == "__main__":
    upsert_paraFunc(TXT_PATH)

    print("\n검색 예")
    searchFunc("소나기의 기원", k=3)